# P03 Database Consistency Audit

**TL;DR.** This notebook runs the repository's read-only P01/P02/P03 consistency audit against `data/tracker.db`. The acceptance threshold is zero critical/high failures; historical immutable parser anomalies remain visible as warnings.

## Context and method

The audit checks the formal database path, the 32/9/5 approved entity inventory, SQLite integrity and foreign keys, Fact lineage, Project/Asset Unit parent consistency, Company-level fallbacks, Information Point membership, and absence of P3-created entities. A yfinance timezone cache is disclosed separately and is not classified as an application database.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

root = Path.cwd().resolve()
if not (root / 'scripts' / 'audit_p3_database_consistency.py').exists():
    root = Path.cwd().resolve().parents[1]
command = [sys.executable, str(root / 'scripts' / 'audit_p3_database_consistency.py'), '--json']
completed = subprocess.run(command, cwd=root, check=True, capture_output=True, text=True, encoding='utf-8')
audit = json.loads(completed.stdout)
print('Source:', audit['database']['runtime_path'])
print('Status:', audit['status'])
print('Summary:', audit['summary'])

In [ ]:
print('Inventory')
for name, count in audit['inventory'].items():
    print(f'{name:28} {count:>6}')
print('\nApplication databases:', audit['database']['database_files'])
print('Non-application caches:', audit['database']['non_application_cache_files'])

In [ ]:
print('Non-passing checks')
for check in audit['checks']:
    if check['status'] != 'pass':
        print(f"{check['status'].upper():8} {check['code']}: {check['description']}")
        print(json.dumps(check['evidence'], ensure_ascii=False, indent=2))

In [ ]:
required_passes = {
    'formal_database_path', 'single_database_file', 'sqlite_integrity',
    'sqlite_foreign_keys', 'p3_fact_foreign_keys',
    'p3_fact_approved_entities', 'latest_fact_entity_consistency',
    'company_layer_attribution', 'latest_fact_lineage',
    'no_p3_entity_autocreation',
}
by_code = {check['code']: check for check in audit['checks']}
assert all(by_code[code]['status'] == 'pass' for code in required_passes)
assert audit['inventory']['stocks'] == 32
assert audit['inventory']['projects'] == 9
assert audit['inventory']['asset_units'] == 5
print('Acceptance assertions passed.')

## Interpretation

The current effective P3 result is entity-consistent and does not create domain entities. The overall status may remain `warning` because immutable historical Parse Runs preserve earlier bad attributions for auditability. Those rows are not silently rewritten or deleted.